# Bluestock Fintech — Mutual Fund Analytics Platform
## Advanced Analytics & Quantitative Risk Metrics
**Team:** SR (Sathwik Ryapaka), AB (Abhiram), AK (Ashmeet Kaur)  
**Date:** 20 November 2026

### Core Analytical Modules
1. **Parametric & Historical Value at Risk (VaR 95%) & Conditional VaR (CVaR / Expected Shortfall)** across all 40 schemes.
2. **Dynamic Rolling 90-Day Sharpe Ratio Analysis** for 5 flagship mutual fund schemes.
3. **Investor Cohort Segmentation Analysis** (2024 vs. 2025 vintage).
4. **SIP Continuity & Churn Risk Analytics** (detection of irregular gaps > 35 days).
5. **Quantitative Fund Recommender Engine** mapping risk appetite to optimal Sharpe allocations.
6. **Sector Concentration Risk (Herfindahl-Hirschman Index — HHI)** across equity portfolios.
7. **5 Executive Quantitative Insights & Strategic Business Takeaways**.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set high-resolution visual styling
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 150

# Database connection
DB_PATH = Path("bluestock_mf.db") if Path("bluestock_mf.db").exists() else Path("../bluestock_mf.db")
conn = sqlite3.connect(DB_PATH)
print(f"Connected to analytical database: {DB_PATH.resolve()}")

## 1. Historical Value at Risk (VaR 95%) & Conditional VaR (CVaR)

### Theoretical Formulation
- **Historical VaR (95%):** The 5th percentile of daily returns: $\text{VaR}_{0.05} = \text{Percentile}(R, 5)$. Represents maximum daily loss with 95% confidence.
- **Conditional VaR (CVaR / Expected Shortfall):** The conditional expectation of returns given that return is below the VaR threshold:
  $$\text{CVaR}_{0.05} = \mathbb{E}[R \mid R \le \text{VaR}_{0.05}]$$
  Penalises catastrophic fat-tail events.

In [ ]:
# Load historical daily returns
query_nav = """
    SELECT n.amfi_code, f.scheme_name, f.fund_house, f.category, f.sub_category, f.risk_category, n.nav_date, n.daily_return
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    ORDER BY n.amfi_code, n.nav_date;
"""
df_nav = pd.read_sql_query(query_nav, conn)

var_cvar_records = []
for code, grp in df_nav.groupby("amfi_code"):
    rets = grp["daily_return"].iloc[1:].dropna().values
    meta = grp.iloc[0]
    
    var_95 = np.percentile(rets, 5)
    cvar_95 = rets[rets <= var_95].mean()
    ann_vol = rets.std() * np.sqrt(252)
    
    var_cvar_records.append({
        "amfi_code": str(code),
        "scheme_name": meta["scheme_name"],
        "fund_house": meta["fund_house"],
        "category": meta["category"],
        "sub_category": meta["sub_category"],
        "risk_category": meta["risk_category"],
        "var_95_daily_pct": round(float(var_95), 4),
        "cvar_95_daily_pct": round(float(cvar_95), 4),
        "annualized_volatility_pct": round(float(ann_vol), 4),
        "min_daily_return_pct": round(float(rets.min()), 4),
        "max_daily_return_pct": round(float(rets.max()), 4)
    })

df_var_cvar = pd.DataFrame(var_cvar_records).sort_values("var_95_daily_pct", ascending=True).reset_index(drop=True)
df_var_cvar.to_csv("var_cvar_report.csv", index=False)
print(f"Computed VaR & CVaR for all {len(df_var_cvar)} mutual fund schemes.")

# Display Top 5 Highest Risk (Worst VaR) vs Top 5 Safest (Best VaR)
print("\n--- Top 5 Schemes with Highest Downside Exposure (Worst VaR 95%) ---")
print(df_var_cvar.head(5)[["scheme_name", "sub_category", "risk_category", "var_95_daily_pct", "cvar_95_daily_pct", "annualized_volatility_pct"]].to_string())

print("\n--- Top 5 Capital Preservation Schemes (Safest VaR 95%) ---")
print(df_var_cvar.tail(5)[["scheme_name", "sub_category", "risk_category", "var_95_daily_pct", "cvar_95_daily_pct", "annualized_volatility_pct"]].to_string())

### Visualizing VaR & CVaR Across Mutual Fund Asset Sub-Categories

In [ ]:
plt.figure(figsize=(14, 6))
order = df_var_cvar.groupby("sub_category")["var_95_daily_pct"].mean().sort_values().index
sns.boxplot(data=df_var_cvar, x="sub_category", y="var_95_daily_pct", order=order, palette="coolwarm")
plt.title("Historical Daily VaR (95%) Distribution by Sub-Category", fontsize=14, weight="bold")
plt.xlabel("Fund Sub-Category", fontsize=12)
plt.ylabel("Historical 95% Daily VaR (%)", fontsize=12)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 2. Dynamic Rolling 90-Day Sharpe Ratio Analysis

Static 3-year or 5-year Sharpe ratios often conceal severe mid-cycle drawdowns. We compute dynamic rolling Sharpe over 90 trading days:
$$\text{Sharpe}_{90d} = \frac{\mu_{90}}{\sigma_{90}} \times \sqrt{252}$$
for 5 key flagship schemes: SBI Bluechip, ICICI Bluechip, Nippon Large Cap, Axis Bluechip, Kotak Bluechip.

In [ ]:
target_schemes = {
    "119551": "SBI Bluechip",
    "120503": "ICICI Bluechip",
    "118632": "Nippon Large Cap",
    "119092": "Axis Bluechip",
    "120841": "Kotak Bluechip"
}

codes_str = "','".join(target_schemes.keys())
query_roll = f"""
    SELECT n.amfi_code, f.scheme_name, n.nav_date, n.daily_return
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.amfi_code IN ('{codes_str}')
    ORDER BY n.amfi_code, n.nav_date;
"""
df_roll = pd.read_sql_query(query_roll, conn)
df_roll["nav_date"] = pd.to_datetime(df_roll["nav_date"])

plt.figure(figsize=(14, 7))
for code, label in target_schemes.items():
    sub = df_roll[df_roll["amfi_code"] == code].sort_values("nav_date").copy()
    sub["roll_mean"] = sub["daily_return"].rolling(90).mean()
    sub["roll_std"] = sub["daily_return"].rolling(90).std()
    sub["rolling_sharpe"] = (sub["roll_mean"] / sub["roll_std"]) * np.sqrt(252)
    
    plt.plot(sub["nav_date"], sub["rolling_sharpe"], label=f"{label} ({code})", linewidth=2, alpha=0.85)

plt.axhline(0, color="black", linestyle="--", linewidth=1.2, alpha=0.7, label="Zero Return / Volatility Parity")
plt.axhline(1.0, color="green", linestyle=":", linewidth=1.2, alpha=0.8, label="Good Risk-Adjusted Threshold (Sharpe = 1.0)")
plt.title("Rolling 90-Day Annualized Sharpe Ratio Comparison (5 Key Funds)", fontsize=14, weight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Rolling Sharpe Ratio (Annualized)", fontsize=12)
plt.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig("rolling_sharpe_chart.png", dpi=300)
plt.show()

## 3. Investor Cohort Segmentation Analysis

Investors are grouped by their **first transaction year** (Cohort 2024 vs. Cohort 2025) to track behavioural retention, capital sizing, and preferred scheme selection.

In [ ]:
query_cohort = """
    SELECT t.investor_id, t.transaction_date, t.amfi_code, f.scheme_name, t.transaction_type, t.amount_inr
    FROM fact_transactions t
    JOIN dim_fund f ON t.amfi_code = f.amfi_code;
"""
df_tx = pd.read_sql_query(query_cohort, conn)
df_tx["tx_date"] = pd.to_datetime(df_tx["transaction_date"])
df_tx["tx_year"] = df_tx["tx_date"].dt.year

# Determine first year per investor
first_tx = df_tx.groupby("investor_id")["tx_year"].min().reset_index()
first_tx.columns = ["investor_id", "cohort_year"]
df_tx = df_tx.merge(first_tx, on="investor_id")

cohort_stats = []
for cohort, c_df in df_tx.groupby("cohort_year"):
    n_inv = c_df["investor_id"].nunique()
    total_inv = c_df["amount_inr"].sum()
    sip_df = c_df[c_df["transaction_type"] == "SIP"]
    avg_sip = sip_df["amount_inr"].mean() if not sip_df.empty else 0
    
    top_fund = c_df.groupby(["amfi_code", "scheme_name"])["amount_inr"].sum().reset_index().sort_values("amount_inr", ascending=False).iloc[0]
    
    cohort_stats.append({
        "cohort_year": cohort,
        "investor_count": n_inv,
        "total_transactions": len(c_df),
        "total_invested_cr": round(total_inv / 1e7, 2),
        "avg_invested_per_investor_inr": round(total_inv / n_inv, 2),
        "avg_sip_amount_inr": round(avg_sip, 2),
        "top_preferred_fund": top_fund["scheme_name"],
        "top_fund_volume_cr": round(top_fund["amount_inr"] / 1e7, 2)
    })

df_cohort_summary = pd.DataFrame(cohort_stats)
df_cohort_summary.to_csv("cohort_analysis.csv", index=False)
df_cohort_summary

## 4. SIP Continuity & Churn Risk Analysis

For retail investors with 6+ SIP transactions, we track the cadence between consecutive SIP deduction dates. When the average gap exceeds **35 days**, the investor is flagged as **At-Risk** of discontinuation/churn.

In [ ]:
query_sip = """
    SELECT investor_id, transaction_date, amount_inr, state, city_tier
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    ORDER BY investor_id, transaction_date;
"""
df_sip_raw = pd.read_sql_query(query_sip, conn)
df_sip_raw["tx_date"] = pd.to_datetime(df_sip_raw["transaction_date"])

sip_investor_records = []
for inv_id, grp in df_sip_raw.groupby("investor_id"):
    if len(grp) >= 6:
        grp = grp.sort_values("tx_date")
        gaps = grp["tx_date"].diff().dt.days.dropna()
        
        avg_gap = gaps.mean()
        max_gap = gaps.max()
        is_at_risk = 1 if avg_gap > 35.0 else 0
        
        sip_investor_records.append({
            "investor_id": inv_id,
            "state": grp.iloc[0]["state"],
            "city_tier": grp.iloc[0]["city_tier"],
            "sip_count": len(grp),
            "total_sip_inr": grp["amount_inr"].sum(),
            "avg_gap_days": round(avg_gap, 2),
            "max_gap_days": int(max_gap),
            "is_at_risk": is_at_risk,
            "status": "At-Risk" if is_at_risk else "Active/Healthy"
        })

df_sip_continuity = pd.DataFrame(sip_investor_records)
df_sip_continuity.to_csv("sip_continuity.csv", index=False)

print(f"Total Investors with 6+ SIP Transactions: {len(df_sip_continuity)}")
print(f"Active / Consistent Investors (Avg Gap <= 35 days): {(df_sip_continuity['is_at_risk'] == 0).sum()} ({(df_sip_continuity['is_at_risk'] == 0).sum() / len(df_sip_continuity) * 100:.2f}%)")
print(f"At-Risk Investors Flagged (Avg Gap > 35 days):        {(df_sip_continuity['is_at_risk'] == 1).sum()} ({(df_sip_continuity['is_at_risk'] == 1).sum() / len(df_sip_continuity) * 100:.2f}%)")

# Breakdown by City Tier
print("\n--- Churn Risk Breakdown by City Tier ---")
print(pd.crosstab(df_sip_continuity["city_tier"], df_sip_continuity["status"], normalize="index") * 100)

## 5. Sector Concentration Risk Analysis (Herfindahl-Hirschman Index — HHI)

The Herfindahl-Hirschman Index (HHI) measures portfolio sector concentration:
$$\text{HHI} = \sum_{i=1}^{K} w_i^2$$
where $w_i$ is the percentage weight of sector $i$.
- $\text{HHI} \ge 2500$: Highly Concentrated Portfolio (elevated sector-specific idiosyncratic risk).
- $1500 \le \text{HHI} < 2500$: Moderately Concentrated.
- $\text{HHI} < 1500$: Well-Diversified Portfolio.

In [ ]:
query_holdings = """
    SELECT h.amfi_code, f.scheme_name, f.fund_house, f.sub_category, h.sector, h.weight_pct
    FROM fact_portfolio_holdings h
    JOIN dim_fund f ON h.amfi_code = f.amfi_code
    WHERE f.category = 'Equity';
"""
df_holdings = pd.read_sql_query(query_holdings, conn)

hhi_records = []
for code, grp in df_holdings.groupby("amfi_code"):
    meta = grp.iloc[0]
    sector_weights = grp.groupby("sector")["weight_pct"].sum()
    hhi = (sector_weights ** 2).sum()
    
    hhi_records.append({
        "amfi_code": str(code),
        "scheme_name": meta["scheme_name"],
        "fund_house": meta["fund_house"],
        "sub_category": meta["sub_category"],
        "hhi_score": round(float(hhi), 2),
        "top_sector": sector_weights.idxmax(),
        "top_sector_weight_pct": round(float(sector_weights.max()), 2),
        "num_sectors": len(sector_weights)
    })

df_hhi = pd.DataFrame(hhi_records).sort_values("hhi_score", ascending=False).reset_index(drop=True)
df_hhi.to_csv("sector_hhi.csv", index=False)

print("--- Top 5 Most Concentrated Equity Portfolios (Highest HHI) ---")
print(df_hhi.head(5)[["scheme_name", "sub_category", "hhi_score", "top_sector", "top_sector_weight_pct"]].to_string())

print("\n--- Top 5 Most Diversified Equity Portfolios (Lowest HHI) ---")
print(df_hhi.tail(5)[["scheme_name", "sub_category", "hhi_score", "top_sector", "top_sector_weight_pct"]].to_string())

## 6. Simple Fund Recommendation Engine

We test the `recommender.py` module across different client risk profiles (Low, Moderate, High).

In [ ]:
from recommender import recommend_funds

for profile in ["Low", "Moderate", "High"]:
    print(f"\n{'='*70}")
    print(f"OPTIMAL RECOMMENDATIONS FOR {profile.upper()} RISK PROFILE")
    print('='*70)
    rec_df = recommend_funds(profile, top_n=3)
    for _, r in rec_df.iterrows():
        print(f"#{r['rank']} | {r['scheme_name']} ({r['sub_category']}) | Sharpe: {r['sharpe_ratio']:.2f} | 3Yr CAGR: {r['return_3yr_pct']:.2f}% | Fee: {r['expense_ratio_pct']:.2f}%")
        print(f"   Rationale: {r['investment_rationale']}")

## 7. Advanced Quantitative Insights & Executive Conclusions

### Insight 1: Heavy Tail Risk & Downside Exposure in Small-Cap Funds
- **Quantitative Finding:** Small Cap schemes (e.g., SBI Small Cap Direct `119599`, Axis Small Cap `119095`) exhibit the steepest daily VaR (95%) of **-2.69%** and Conditional VaR (Expected Shortfall) of **-3.24%**, alongside annualized volatility exceeding **25.0%**.
- **Portfolio Implication:** While small caps yield the highest 3-year returns (23.39%), their tail risk requires investors to maintain at least a 5-to-7 year investment horizon and dynamic asset allocation to absorb drawdowns.

### Insight 2: Regime-Dependent Rolling Sharpe Dynamics
- **Quantitative Finding:** Large-cap flagship funds (e.g., SBI Bluechip `119551`, ICICI Bluechip `120503`) show severe rolling 90-day Sharpe volatility, swinging from a low of **-3.64** during market pullbacks to **+6.77** during momentum rallies.
- **Portfolio Implication:** Static point-in-time Sharpe ratios published in fund factsheets obscure intermediate volatility shocks. Wealth advisors must track multi-window rolling Sharpe to evaluate fund manager consistency across bull and bear cycles.

### Insight 3: Investor Vintage & Capital Concentration (Cohort Dynamics)
- **Quantitative Finding:** The **2024 Cohort** represents **99.1%** of cumulative capital invested (**₹349.11 Crore** across 4,803 investors) with UTI Nifty 50 Index (`102885`) as the anchor allocation.
- **Portfolio Implication:** The **2025 Cohort**, while smaller in count (197 investors, ₹3.05 Crore), features a **22.8% higher average SIP commitment** (₹13,505 vs ₹10,996), indicating rising ticket sizes among newer fintech-native investors targeting high-beta small cap strategies (`119599`).

### Insight 4: Structural SIP Drop-Off & Churn Vulnerability
- **Quantitative Finding:** Among 1,362 investors with 6+ SIP deductions, **97.8% (1,332 investors)** exhibit an average gap between SIP dates greater than 35 days (often averaging 40 to 85 days).
- **Strategic AMC Recommendation:** This reveals high friction in manual payments (UPI/Cheques). AMCs must push automated e-NACH mandates and UPI Autopay to eliminate installment gaps and improve compounding retention.

### Insight 5: Portfolio Concentration & Sectoral Vulnerability (HHI Metric)
- **Quantitative Finding:** **Axis Bluechip Fund (`119092`)** carries the highest sector concentration score (**HHI = 2,967.69**), with **48.69% of its equity weight locked into the Information Technology sector**.
- **Risk Takeaway:** Funds with HHI > 2500 face significant idiosyncratic drawdown risk if their dominant sector experiences headwinds. By contrast, schemes like SBI Bluechip (`119551`) maintain an HHI under 1,800, providing balanced cross-sector insulation across Banking, IT, Pharma, and Automobile sectors.